# 06 · Test değerlendirmesi (Adım 8) — TEK TUR, ön-kayıtlı
**Ön-kayıt:** `docs/adim8_tam_egitim_onkayit.md` §4 ve §6. Bu not defteri test bölmesini (`--allow-test`) açan **tek** yerdir. Tüm eğitimler (S1 3 seed, S2, tetikleyici katlaması) bitmeden ve tetikleyici sonucu kaydedilmeden çalışmaz.

**Kaggle ayarları:** GPU T4 x2 · Internet On · Secrets `GITHUB_TOKEN` Attached · Input: dataset `melihkotman/disasterm3-turkiye` **ve** tam eğitim not defterinin çıktısı (*Add Input → Notebooks → tam eğitim not defterin → Output*).

**Tek tur kuralı:** sonuç görüldükten sonra yapılandırma, checkpoint veya eşik değiştirilmez; yeniden koşu yalnız teknik hata içindir (kesinti, OOM).


In [ ]:
# ---- PARAMETRELER ----
RUN_TEST = False             # True yapmadan test'e dokunulmaz (kapı)
CODE_REF = "adim8-test-v1"   # değerlendirme kodu etiketi (eğitim: adim8-full-v1; fark: yalnız per-tile sayaç + rapor betiği)
MODELS = {"s1_seed42": "S1", "s1_seed43": "S1", "s1_seed44": "S1", "s2_seed42": "S2"}   # model adı -> senaryo
ROLES = ("test", "control")                 # test: Antakya+e6+e7 (S1) / Osmaniye+e7 (S2); control: Gaziantep (hasarsız, yanlış alarm denetimi)
print("RUN_TEST:", RUN_TEST, "| CODE_REF:", CODE_REF)


In [ ]:
# 1) Yardımcılar + kurulum (ever-beta 0.5.6 sabit!)
import os, sys, json, time, glob, hashlib, subprocess, threading, shutil

def sh(cmd, check=True, mask=(), tail=3000):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    out = (r.stdout or "") + (r.stderr or "")
    for m in mask:
        out = out.replace(m, "***")
    if out.strip():
        print(out.strip()[-tail:])
    if check and r.returncode:
        raise SystemExit(f"komut hata verdi (çıkış {r.returncode}); çıktıya bak")
    return r.returncode

sh('pip install -q timm "ever-beta==0.5.6" albumentations tifffile einops "datasets[vision]"')
sh("pip install -q --no-deps git+https://github.com/Z-Zheng/pytorch-change-models.git")
sh("nvidia-smi --query-gpu=index,name,memory.total --format=csv")


In [ ]:
# 2) Kodu al, etikete sabitle, sürümü kaydet
REPO = "/kaggle/working/deprem-vlm-rag"

def get_code():
    if os.path.exists(REPO):
        return
    tok = None
    try:
        from kaggle_secrets import UserSecretsClient
        tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception as e:
        print("GITHUB_TOKEN okunamadı:", type(e).__name__)
    if tok:
        rc = sh(f"git clone -q https://{tok}@github.com/MelihKotman/Deprem-VLM-RAG.git {REPO}", check=False, mask=(tok,))
        if rc == 0:
            sh(f"git -C {REPO} remote set-url origin https://github.com/MelihKotman/Deprem-VLM-RAG.git")
    if not os.path.exists(REPO):
        cand = glob.glob("/kaggle/input/**/deprem-vlm-rag", recursive=True)
        if not cand:
            raise SystemExit("kod alınamadı: GITHUB_TOKEN 'Attached' mı? ya da 'kod' dataset'ini ekle")
        shutil.copytree(cand[0], REPO)
        print("UYARI: kod 'kod' dataset'inden alındı (git etiketi doğrulanamaz)")

get_code()
git_sha = None
if os.path.exists(f"{REPO}/.git"):
    sh(f"git -C {REPO} fetch -q --tags", check=False)
    if sh(f"git -C {REPO} checkout -q {CODE_REF}", check=False):
        raise SystemExit(f"CODE_REF '{CODE_REF}' bulunamadı: yerelde `git tag {CODE_REF}` + `git push origin {CODE_REF}` yap")
    git_sha = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
files = sorted(glob.glob(f"{REPO}/src/*.py")) + [f"{REPO}/config/base.yaml"]
src_sha1 = hashlib.sha1(b"".join(open(f, "rb").read() for f in files)).hexdigest()
pipv = subprocess.run("pip freeze 2>/dev/null | grep -i -E '^(torch|torchvision|ever-beta|torchange|timm|numpy|opencv)'", shell=True, capture_output=True, text=True).stdout
with open("/kaggle/working/code_version.txt", "w") as f:
    f.write(f"code_ref={CODE_REF}\ngit_sha={git_sha}\nsrc_config_sha1={src_sha1}\n{pipv}")
print(open("/kaggle/working/code_version.txt").read())
os.chdir(REPO); sys.path.insert(0, REPO)
sh("python -m src.config && python -m src.dm3_verify", tail=1500)
sh("python -m src.selftest_eval", tail=800)


In [ ]:
# 3) Koşu yardımcıları (alt süreç + günlük dosyası; ekrana yalnız ilerleme/hata satırları)
RUNS = "/kaggle/working/runs"; LOGS = "/kaggle/working/logs"
os.makedirs(RUNS, exist_ok=True); os.makedirs(LOGS, exist_ok=True)
CACHE_ROOT = "/kaggle/temp" if os.path.isdir("/kaggle/temp") else "/tmp"   # çıktıya (Output) girmesin
print("önbellek kökü:", CACHE_ROOT); sh(f"df -h {CACHE_ROOT} | tail -1")
SHOW = ("İLERLEME", "Traceback", "Error", "error", "timing:", "tetikleyici:", "metrikler", "out of memory", "EXTRAPOLATION", "[S1]", "[S2]", "--max-minutes", "kilitli", "ayrılmıştır")

def run_job(tag, args, gpu, cache, workers, module="src.train_changeos"):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED="1")
    cmd = [sys.executable, "-m", module, *args, "--cache-dir", cache, "--workers", str(workers)]
    oom = False
    with open(f"{LOGS}/{tag}.log", "a") as log:
        p = subprocess.Popen(cmd, cwd=REPO, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            log.write(line)
            if "out of memory" in line.lower():
                oom = True
            if any(k in line for k in SHOW):
                print(f"[{tag}] {line.rstrip()[:260]}", flush=True)
        p.wait()
    return p.returncode, oom

def train_job(job, gpu, cache, workers):
    name, scen, seed, trig = job
    out = f"{RUNS}/{name}"
    if os.path.exists(f"{out}/model_final.pt") and os.path.exists(f"{out}/metrics.json"):
        print(f"[{name}] zaten tamam, atlanıyor"); return 0
    base = ["--scenario", scen, "--epochs", str(EPOCHS), "--seed", str(seed), "--max-minutes", "600", "--out", out]
    if trig:
        base += ["--holdout", "Osmaniye", "--trigger-fold"]
    extra = []
    for attempt in range(3):
        args = base + extra + (["--resume"] if os.path.exists(f"{out}/last.pt") else [])
        rc, oom = run_job(name, args, gpu, cache, workers)
        if rc == 0:
            return 0
        if rc == 3:                       # --max-minutes: kaldığı yerden devam
            continue
        if oom and "--no-cudnn-benchmark" not in extra:
            extra = ["--no-cudnn-benchmark"]; print(f"[{name}] OOM → --no-cudnn-benchmark ile yeniden"); continue
        break
    print(f"[{name}] BAŞARISIZ (çıkış {rc}); günlük: {LOGS}/{name}.log")
    return rc

def run_lane(lane, gpu, workers):
    cache = f"{CACHE_ROOT}/cache_{lane}"
    for job in LANES[lane]:
        t = time.time(); rc = train_job(job, gpu, cache, workers)
        print(f"[şerit {lane}] {job[0]} bitti: çıkış {rc}, {(time.time() - t) / 60:.0f} dk", flush=True)


In [ ]:
# 4) Kapı: tüm eğitimler + tetikleyici hazır mı? Checkpoint'leri bul
def find(name, fn):
    c = glob.glob(f"/kaggle/input/**/runs/{name}/{fn}", recursive=True)
    return c[0] if c else None
CK = {m: find(m, "model_final.pt") for m in MODELS}
TRIG = find("trigger_osmaniye_seed42", "metrics.json")
print("checkpoint'ler:", {k: bool(v) for k, v in CK.items()}, "| tetikleyici metrics:", bool(TRIG))
if TRIG:
    print("tetikleyici (kayıtlı):", json.load(open(TRIG)).get("trigger"))
READY = all(CK.values()) and bool(TRIG)
if not READY:
    print("EKSİK: tüm checkpoint'ler ve tetikleyici sonucu girdi olarak ekli olmalı → test çalıştırılmaz")
if RUN_TEST and not READY:
    raise SystemExit("kapı kapalı")


In [ ]:
# 5) TEK TUR test + kontrol değerlendirmesi (iki GPU, iş kuyruğu)
TR = "/kaggle/working/test_runs"; os.makedirs(TR, exist_ok=True)
JOBS = [(m, MODELS[m], role) for m in MODELS for role in ROLES]
q = list(JOBS); lock_q = threading.Lock(); RES = {}
def worker(gpu):
    cache = f"{CACHE_ROOT}/cache_eval{gpu}"
    while True:
        with lock_q:
            if not q: return
            m, scen, role = q.pop(0)
        out = f"{TR}/{m}_{role}"
        if os.path.exists(f"{out}/per_tile.csv"):
            print(f"[{m}_{role}] zaten var, atlanıyor"); continue
        args = ["--scenario", scen, "--eval-only", "--eval-role", role, "--ckpt", CK[m], "--out", out]
        if role == "test":
            args.append("--allow-test")
        t = time.time(); rc, oom = run_job(f"{m}_{role}", args, gpu, cache, 2)
        RES[f"{m}_{role}"] = rc
        print(f"[{m}_{role}] bitti: çıkış {rc}, {(time.time() - t) / 60:.1f} dk", flush=True)
if RUN_TEST and READY:
    t0 = time.time()
    th = [threading.Thread(target=worker, args=(g,)) for g in (0, 1)]
    [t.start() for t in th]; [t.join() for t in th]
    print("TOPLAM:", f"{(time.time() - t0) / 60:.1f} dk", RES)
else:
    print("RUN_TEST=False: test koşulmadı")


In [ ]:
# 6) Ön-kayıtlı rapor (karo bazlı bootstrap %95 AA; sınıf bazlı F1; ikili hasar var/yok) — yalnız tüm işler tamamsa
def pt(m, role): return f"{TR}/{m}_{role}/per_tile.csv"
if RUN_TEST and READY:
    ok = all(os.path.exists(pt(m, r)) for m in MODELS for r in ROLES)
    print("tüm per_tile.csv mevcut:", ok)
    if ok:
        s1 = [f"{m}={pt(m, 'test')}" for m in MODELS if MODELS[m] == "S1"]
        sh(f"python -m src.test_report --runs {' '.join(s1)} --out /kaggle/working/rapor_s1_test", tail=4000)
        sh(f"python -m src.test_report --runs s2_seed42={pt('s2_seed42', 'test')} --out /kaggle/working/rapor_s2_test", tail=2500)
        ctl = [f"{m}_kontrol={pt(m, 'control')}" for m in MODELS]
        sh(f"python -m src.test_report --runs {' '.join(ctl)} --out /kaggle/working/rapor_kontrol", tail=2500)
        sh(f"python -m src.summarize_runs {TR}", check=False, tail=5000)
        print(open("/kaggle/working/rapor_s1_test/rapor.md").read()[:6000])
